# Community Needs Indexing - pull, clean, export

Indicator: SNAP / food stamp receipt rate (percent of households), ACS 5-year Subject Table S2201,
variable S2201_C04_001E. Geographies: Allegheny County, Pennsylvania, United States.

Run by hand a few times a year. Overwrites `/data/community_needs_index.json`.

**API key:** set the `CENSUS_API_KEY` environment variable before running - do not paste the key into
this notebook. In a Codespace: repo Settings -> Secrets and variables -> Codespaces -> add `CENSUS_API_KEY`,
or run `export CENSUS_API_KEY=your_key` in the terminal for a one-off session.

In [1]:
import json
import os
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests

API_KEY = os.environ.get("CENSUS_API_KEY")
if not API_KEY:
    raise RuntimeError("CENSUS_API_KEY not set. Set it as an environment variable, do not hardcode it here.")

RAW_DIR = Path("../raw")
DATA_DIR = Path("../data")
RAW_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

today = date.today().isoformat()
print("Pipeline run date:", today)

Pipeline run date: 2026-10-04


In [ ]:
VARIABLE = "S2201_C04_001E"  # Percent households receiving food stamps/SNAP, total
DATASET = "acs/acs5/subject"

# ACS5 vintages to pull. Edit this list - add the current year once it's released,
# drop early years if the table isn't available that far back for your dataset.
YEARS = [2014, 2016, 2018, 2020, 2022]

GEOGRAPHIES = {
    "Allegheny County": {"for": "county:003", "in": "state:42"},
    "Pennsylvania": {"for": "state:42"},
    "United States": {"for": "us:1"},
}

## 1. Pull

One request per (year, geography). Caches every raw response, dated, before any parsing.

In [ ]:
rows = []
raw_responses = {}

for year in YEARS:
    url = f"https://api.census.gov/data/{year}/{DATASET}"
    for geo_name, geo_params in GEOGRAPHIES.items():
        params = {"get": f"NAME,{VARIABLE}", "key": API_KEY, **geo_params}
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            payload = resp.json()
        except Exception as exc:
            print(f"FAILED  year={year}  geo={geo_name}  error={exc}")
            continue

        raw_responses[f"{year}_{geo_name}"] = payload

        header, data_row = payload[0], payload[1]
        record = dict(zip(header, data_row))
        value_raw = record.get(VARIABLE)

        # Census uses sentinel negative codes (e.g. -666666666) for missing/suppressed data
        value = None
        if value_raw is not None:
            try:
                value_f = float(value_raw)
                value = None if value_f < 0 else value_f
            except ValueError:
                value = None

        rows.append({"year": year, "geography": geo_name, "value": value})
        time.sleep(0.2)  # be polite to the API

with open(RAW_DIR / f"community_needs_index_raw_{today}.json", "w") as f:
    json.dump(raw_responses, f, indent=2)

df = pd.DataFrame(rows)
df

## 2. Check for gaps

If any (year, geography) pair is missing or null, decide whether to drop that year or leave a gap -
the chart draws through gaps (`spanGaps: true`) rather than breaking, but don't silently ship a hole
without knowing it's there.

In [ ]:
expected = len(YEARS) * len(GEOGRAPHIES)
print(f"Expected {expected} rows, got {len(df)}")
print(df[df["value"].isna()])

## 3. Export

Pivot to the wide schema the site expects and overwrite `/data/community_needs_index.json`.

In [ ]:
wide = df.pivot(index="year", columns="geography", values="value").reset_index()
series = wide.to_dict(orient="records")

payload = {
    "indicator": "SNAP / food stamp receipt rate (% of households)",
    "geographies": list(GEOGRAPHIES.keys()),
    "unit": "percent",
    "snapshot_as_of": today,
    "status": "live",
    "source": f"U.S. Census Bureau, American Community Survey 5-Year Estimates, Subject Table S2201, variable {VARIABLE}",
    "series": series,
    "notes": "",
}

with open(DATA_DIR / "community_needs_index.json", "w") as f:
    json.dump(payload, f, indent=2)

print("Wrote", DATA_DIR / "community_needs_index.json")
payload